# 02. Treinar Splink (dedupe cross-source)

Profile, blocking pré-treino, treino do modelo, predict e clustering.

A coorte não entra aqui: o modelo é treinado sem ver os rótulos, e a avaliação
roda em [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb) sobre o modelo salvo.

**EDA descritiva:** [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).

In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_LIMPA,
    USE_PHONETIC_STRIP_VOWELS,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 100_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra profile/blocking: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}')


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.

In [ ]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'idade', 'cep', 'sexo', 'uf',
]
cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


## Exploração pré-modelo

Profile Splink das colunas de linkage e análise de blocking (cumulativo + maiores blocos).

In [ ]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns, n_largest_blocks

# Blocking rules — colunas completas (sem substr)
blocking_rules = [
    block_on('primeiro_nome', 'ultimo_nome'),
    block_on('ultimo_nome', 'data_nascimento'),
    block_on('primeiro_nome', 'data_nascimento'),
    block_on('cep', 'ultimo_nome'),
    block_on('cep', 'primeiro_nome'),
    block_on('nome_mae', 'data_nascimento'),
]

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome', 'ultimo_nome', 'nome_completo_phon',
        'cep', 'data_nascimento', 'idade',
    ],
)

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=analysis_table,
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='dedupe_only',
)

for rule in blocking_rules:
    print(rule)
    n_largest_blocks(
        table_or_tables=analysis_table,
        blocking_rule=rule,
        db_api=db_api,
        n=5,
    )

## Modelo Splink

Settings e `Linker` sobre a base completa (`SPLINK_INPUT_VIEW`).

In [ ]:
from splink import Linker, SettingsCreator
import splink.comparison_library as cl

input_cols = set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)

comparisons = [
    cl.NameComparison('nome_completo'),
    cl.NameComparison('primeiro_nome').configure(term_frequency_adjustments=True),
    cl.NameComparison('ultimo_nome').configure(term_frequency_adjustments=True),
    cl.NameComparison('nome_completo_phon'),
    cl.DateOfBirthComparison('data_nascimento', input_is_string=True),
    cl.ExactMatch('idade').configure(term_frequency_adjustments=True),
    cl.NameComparison('nome_mae'),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    cl.ExactMatch('uf').configure(term_frequency_adjustments=True),
]
if USE_PHONETIC_STRIP_VOWELS and 'nome_completo_phon_sv' in input_cols:
    comparisons.append(cl.NameComparison('nome_completo_phon_sv'))

settings = SettingsCreator(
    link_type='dedupe_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
)
linker = Linker(SPLINK_INPUT_VIEW, settings, db_api=db_api)

In [ ]:
deterministic_rules = [
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento'),
    block_on('nome_mae', 'data_nascimento'),
]
linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.8)
linker.training.estimate_u_using_random_sampling(max_pairs=1_000_000)
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento')
)

SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).

In [ ]:
linker.visualisations.match_weights_chart()
linker.evaluation.unlinkables_chart()

## Predict + clustering

In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).

In [ ]:
records_sample = df_predict.as_pandas_dataframe(limit=5).to_dict(orient='records')
linker.visualisations.waterfall_chart(records_sample, filter_nulls=False)

## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).

In [ ]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))

## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Precision/recall contra a
coorte ficam no NB03, sobre o modelo salvo acima.

In [ ]:
display(df_predictions['match_probability'].describe())
faixas = pd.cut(df_predictions['match_weight'], bins=20)
display(
    df_predictions.groupby(faixas, observed=True)
    .size()
    .rename('n_pares')
    .to_frame()
)

In [ ]:
tamanhos = df_clusters.groupby('cluster_id').size()
print(f'Clusters: {tamanhos.size:,} | maior: {tamanhos.max():,} | singletons: {(tamanhos == 1).sum():,}')
display(
    tamanhos.value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)

# Clusters grandes demais indicam blocking/threshold frouxo — inspecionar antes do NB03.
display(tamanhos.sort_values(ascending=False).head(10).rename('tamanho').to_frame())

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(SPLINK_PREDICTIONS)
df_clusters.to_parquet(SPLINK_CLUSTERS)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)

## Encerrar

Artefatos prontos para o [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb).

In [ ]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')

con.close()
